In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_departments
# Source          : departments.csv
# Target          : procurement.silver.silver_departments
# Audit Table     : procurement.audit.duplicate_departments
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned department master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read Department data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
# ============================================================
# Read Bronze Department Table
# ============================================================

bronze_departments_df = read_delta(BRONZE_DEPARTMENTS)

preview(bronze_departments_df,"Bronze Departments")

In [0]:
# ============================================================
# Remove Duplicate Department Records
# Business Rule:Keep the first occurrence of each Department ID.
# ============================================================
window_spec = Window.partitionBy("department_id").orderBy("department_id")

silver_departments_df = (
    bronze_departments_df
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
)

preview(
    silver_departments_df,
    "After Removing Duplicates"
)

In [0]:
# ============================================================
# Standardize Text Columns
# ============================================================

from pyspark.sql.functions import trim, initcap

silver_departments_df = (
    silver_departments_df
        .withColumn(
            "department_name",
            initcap(trim(col("department_name")))
        )
        .withColumn(
            "division",
            initcap(trim(col("division")))
        )
        .withColumn(
            "region",
            initcap(trim(col("region")))
        )
        .withColumn(
            "cost_center",
            upper(trim(col("cost_center")))
        )
)

preview(silver_departments_df,"Silver Departments")

In [0]:
# ============================================================
# Business Rule Validation
# ============================================================

invalid_budget = silver_departments_df.filter(col("annual_budget_usd") < 0)

print(f"Invalid Budget Records : {invalid_budget.count()}")

display(invalid_budget)

In [0]:
# ============================================================
# Add Silver Metadata
# ============================================================

silver_departments_df = (silver_departments_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_departments_df,table_name=SILVER_DEPARTMENTS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

print("=" * 60)
print("Silver Department Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records : {bronze_departments_df.count()}")
print(f"Silver Records : {silver_departments_df.count()}")
print(f"Duplicates Removed : {bronze_departments_df.count() - silver_departments_df.count()}")